In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

model = init_chat_model(model_provider="google_genai", 
                        model ="gemini-2.5-flash",
                        temperature=0.7,
                        max_output_tokens=1024,
                        top_p=0.95,
                        top_k=40,
                        timeout = 30,
                        max_retries = 3,
                        load_env=True)

Unexpected argument 'load_env' provided to ChatGoogleGenerativeAI.
/home/lakshay/LangChain/.venv/lib/python3.12/site-packages/langchain/chat_models/base.py:540: UserWarning: WARNING! load_env is not default parameter.
                load_env was transferred to model_kwargs.
                Please confirm that load_env is what you intended.
  return creator_func(model=model, **kwargs)


In [2]:
from langchain_core.tools import tool

@tool
def calculator(expression:str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(expression))    

In [5]:
result = calculator.invoke("25 * 5 + 10")

print(result)

135


In [8]:
model_with_tools = model.bind_tools([calculator])

In [11]:
from langchain.messages import SystemMessage, HumanMessage , ToolMessage

syst_message = SystemMessage(content="You are a helpful assistant that can perform calculations.")
human_message = HumanMessage(content="What is 25 * 5 + 10?")

messages = [syst_message, human_message]

ai_message = model_with_tools.invoke(messages)

tool_call = ai_message.tool_calls[0]
tool_result = calculator.invoke(tool_call['args'])

tool_message = ToolMessage(content=tool_result,tool_call_id=tool_call["id"],)

final_result = model_with_tools.invoke(
    messages + [ai_message, tool_message]
)

print(final_result.content)

The result is 135.
